# ParFlow grid mapping from latitude/longitude to grid cells

In this notebook, we'll explore in more detail how site locations (stream gages, groundwater wells, SNOTEL stations, etc.) were mapped to the ParFlow-CONUS grid. These result in the `conus1_i`, `conus1_j`, `conus2_i` and `conus2_j` grid mappings pre-loaded into the pandas DataFrame that gets returned by `get_point_metadata`. The mapping process differs between stream gages and other site locations. We'll take a look at both.

### Set Up

In [1]:
import numpy as np
import hf_hydrodata as hf

from pyproj import CRS, Transformer

You'll need to register for a HydroFrame Account and PIN in order to use the `hf_hydrodata` package. The [Getting Started page](https://hf-hydrodata.readthedocs.io/en/latest/getting_started.html) has full details on how to sign up for an account and set up a PIN.

Once you create a PIN, you'll need to register your PIN.

In [ ]:
hf.register_api_pin("your_email", "your_api_pin")

### Example: Mapping latitude/longitude coordinates to the ParFlow-CONUS2 grid

This example represents our mapping procedure for site locations such as groundwater wells, SNOTEL stations, SCAN stations, etc. This does not fully represent our mapping procedure for stream gages; see the next example for more details on that approach.

The following code is packaged into the `hf_hydrodata` function `to_ij`. Let's start by seeing how that operates.

In [2]:
swe_station = hf.get_point_metadata(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod", site_ids="1168:NM:SNTL")

conus2_i, conus2_j = hf.to_ij("conus2", swe_station.loc[0, "latitude"], swe_station.loc[0, "longitude"])
print(conus2_i, conus2_j)

# See that this calculation matches our pre-computed i/j grid mapping
swe_station.loc[:, ["site_id", "site_name", "latitude", "longitude", "conus2_i", "conus2_j"]]


1471 1337


,site_id,site_name,latitude,longitude,conus2_i,conus2_j
0,1168:NM:SNTL,Taos Powderhorn,36.58195,-105.45617,1471.0,1337.0


In `hf.to_ij`, the package uses a custom, precomputed Lambert Conformal Conic projection to convert lat/lon into grid indices. This is significantly faster than using generic projection engines like `pyproj` and is important for our [HydroGEN](https://hydrogen.princeton.edu/) user interface performance. 

In this example notebook, however, we demonstrate the standard `pyproj`-based conversion workflow so users can generalize the method to any coordinate system or grid, not only those supported by the HydroFrame project.

In [ ]:
# Get grid metadata for the ParFlow-CONUS2 grid
conus2_grid_data = hf.get_table_row("grid", id="conus2")
print(conus2_grid_data)

{'id': 'conus2', 'title': 'conus 2', 'shape': [10, 3256, 4442], 'latlng_bounds': [21.8170599154073, -126.88755692881833, 53.20274381640737, -64.7677149695924], 'resolution_meters': '1000', 'crs_name': '', 'origin': [-2208000.30881173, -1668999.65483222], 'crs': ' +proj=lcc +lat_1=30 +lat_2=60 +lon_0=-97.0 +lat_0=40.0000076294444 +a=6370000.0 +b=6370000'}


In [ ]:
# Set up pyproj transformer to convert lat/lon to ParFlow-CONUS2 projection coordinates
crs_geog = CRS.from_epsg(4326)
crs_proj = CRS.from_string(conus2_grid_data['crs'])

to_proj = Transformer.from_crs(crs_geog, crs_proj, always_xy=True)

In [ ]:
# Transformation for a single point (the location of our SNOTEL station)
lon, lat = -105.45617, 36.58195

# 1) Convert to projected coordinates. For ParFlow-CONUS2, these are in meters.
x_m, y_m = to_proj.transform(lon, lat)

# 2) Remove false origin if your grid uses one
false_easting  = float(conus2_grid_data['origin'][0])
false_northing = float(conus2_grid_data['origin'][1])

x_m -= false_easting
y_m -= false_northing

# 3) Use grid resolution to convert to grid units
x = x_m / int(conus2_grid_data['resolution_meters'])
y = y_m / int(conus2_grid_data['resolution_meters'])

print(int(np.floor(x)), int(np.floor(y)))

1471 1337


### Example: Mapping latitude/longitude coordinates to the ParFlow-CONUS2 stream network

This example represents our mapping procedure for stream gages. We start with the same `to_ij` approach as described above for other site locations. However, to ensure that gages are mapped to the same grid cells as are identified as having a stream, we do some additional processing steps.


We'll select a stream gage and use the `to_ij` function grid coordinate results as our starting location.

In [3]:
streamgage_metadata = hf.get_point_metadata(dataset="usgs_nwis", variable="streamflow",
                                            temporal_resolution="daily", aggregation="mean",
                                            site_ids="01401000")

streamgage_metadata.loc[:, ["site_id", "site_name", "latitude", "longitude"]]

,site_id,site_name,latitude,longitude
0,01401000,Stony Brook at Princeton NJ,40.333056,-74.681944


In [4]:
# Starting location of the streamgage in the ParFlow-CONUS2 grid
starting_i, starting_j = hf.to_ij("conus2", streamgage_metadata.loc[0, "latitude"], streamgage_metadata.loc[0, "longitude"])
print(starting_i, starting_j)

4018 1958


For stream gages, we start from the initial latitude/longitude mapping. Then we look a radius of 3 grid cells in each direction away from the starting location and see whether any of the other locations has a better-matched drainage area comparison between the USGS-reported drainage area and the ParFlow drainage area. The idea is that sometimes the pure latitude/longitude mapping does not exactly align with where the ParFlow model has identified as grid cells where a stream is. This adjustment process aims to better align their placement.

First, let's convert the USGS drainage area field from square miles into square kilometers to match the units we have from ParFlow.

In [5]:
streamgage_metadata["usgs_drainage_area_sqkm"] = streamgage_metadata["usgs_drainage_area"] * 2.59

Next we will request the ParFlow-CONUS2 drainage area to compare against.

In [44]:
# Read in ParFlow CONUS2 drainage area file.
parflow_drainage_area = hf.get_gridded_data(dataset="conus2_domain", variable="drainage_area").squeeze()
print(parflow_drainage_area.shape)

(3256, 4442)


The USGS drainage area is reported in square miles. Let's convert to square kilometers for consistency.

In [10]:
# Extract USGS drainage area (sq km)
usgs_da = streamgage_metadata.loc[streamgage_metadata["site_id"] == "01401000", "usgs_drainage_area_sqkm"].item()
print(usgs_da)

115.255


In [45]:
# The ParFlow-CONUS2 grid indexes from the lower left corner. From hf_hydrodata, that is the (0,0) NumPy index. 
# So we can directly use the starting_i and starting_j values as the center of our radius to extract a subset of the ParFlow drainage area array. 

# Get a subset of ParFlow drainage area array of a radius around the site cell in question
radius=3
d_radius = parflow_drainage_area[(starting_j-radius):(starting_j+radius+1), (starting_i-radius):(starting_i+radius+1)]
print(d_radius)

[[  3.   9.  10. 119.   2. 106.   4.]
 [  3.   5.   1. 108.   1. 101. 100.]
 [  1.   1.   1. 106.  10.   8.  97.]
 [  1.   1.  92.  94.   1.   7.  94.]
 [ 72.   1.  90.   1.   1.   6.   4.]
 [ 74.  75.  88.   1.   1.   3.   4.]
 [  2.   4.  11.   4.   1.   1.   1.]]


The next code section finds the location of the grid cell with the drainage area closest to the USGS-reported drainage area. Note that the ParFlow drainage area at our "starting guess" is in the middle of `d_radius` and is 94. From above, we know that the USGS-reported drainage area for this stream gage is ~115.3. Looking at `d_radius`, we see that the grid cell in the lower middle with a drainage area of 119 is the closest value to 115.3. Importantly, it is closer than the current location's value of 94. The below algorithm should select that as the new i, j mapping for this stream gage.

In [41]:
# This works when the array is square and has an odd number of rows and columns
assert d_radius.shape[0] == d_radius.shape[1]
assert d_radius.shape[0] % 2 == 1
center_i = d_radius.shape[1] // 2
center_j = d_radius.shape[0] // 2

# Get i and j indices of radius array for where the ParFlow value is closest to the USGS drainage area
(j_pick, i_pick) = np.unravel_index(np.argmin(np.abs(d_radius - usgs_da)), d_radius.shape)

# Calculate offsets of optimal value from center (current i/j)
offset_i = i_pick - center_i
offset_j = j_pick - center_j 

print(f"Offset i: {offset_i}, Offset j: {offset_j}")
print(f"ParFlow drainage area at current location: {d_radius[center_j, center_i]}")
print(f"ParFlow drainage area at optimal location: {d_radius[j_pick, i_pick]}")

Offset i: 0, Offset j: -3
ParFlow drainage area at current location: 94.0
ParFlow drainage area at optimal location: 119.0


One final detail for our processing is that we only change the i/j location of a gage if the calculated drainage area difference between the proposed location and the USGS-reported drainage area is within 20%. We don't move a gage from its latitude/longitude-mapped location if the drainage area difference after the move would be too large.

In [42]:
# Calculate area difference to determine if we want to move location
pct_area_diff = (d_radius[j_pick, i_pick] - usgs_da) / usgs_da
print(f"Percentage difference in drainage area: {pct_area_diff}")

if abs(pct_area_diff) <= 0.2:
    i_new = starting_i + offset_i
    j_new = starting_j + offset_j

print(f"Original i/j location: {starting_i}, {starting_j}")
print(f"New i/j location: {i_new}, {j_new}")

Percentage difference in drainage area: 0.032493167324628036
Original i/j location: 4018, 1958
New i/j location: 4018, 1955


In [43]:
# Confirm this is what is included in the hf_hydrodata metadata
streamgage_metadata.loc[:, ["site_id", "site_name", "latitude", "longitude", "conus2_i", "conus2_j"]]

,site_id,site_name,latitude,longitude,conus2_i,conus2_j
0,01401000,Stony Brook at Princeton NJ,40.333056,-74.681944,4018.0,1955.0


That's it! For the Parflow-CONUS2 domain, we repeated the above process for all USGS stream gages within the hf_hydrodata network to pre-compute the `conus2_i` and `conus2_j` values for users.